# 09 — Cross-Module Consistency

Compare outputs across TechnologyModel, EconomicModel, FinancialCalculator, and Streamlit.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Checklist

- [ ] Technology annual_energy matches economic revenue inputs
- [ ] LCOE from FinancialCalculator ≈ implied LCOE from cost/energy
- [ ] Baseline mean MC LCOE same order of magnitude as baseline
- [ ] Document Streamlit vs engine divergences (`webapp/app.py`)

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.baseline_optimization import BaselineOptimization

config = load_config("alaska")
config.uncertainty["monte_carlo_runs"] = 10
results = BaselineOptimization(config).optimize("production", 200_000, method="scipy")

assert_energy_balance(
    results.technical_metrics["annual_energy"],
    results.technical_metrics["total_capacity"],
    results.technical_metrics["capacity_factor"],
)
assert_cf_bounds(results.technical_metrics["capacity_factor"])

# TODO: reconcile LCOE from financial_metrics vs manual cost/energy calculation
print("Financial:", results.financial_metrics)
print("Technical:", results.technical_metrics)